In [ ]:
import dataclasses
import logging
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.stats import circmean

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from scripts.analysis_common import (  # noqa: E402
    FREQUENCY_BANDS,
    _WAVELET_BAND_FREQ_RESOLUTION_HZ,
    _wavelet_transform,
    analyzers_to_datasets,
    load_analyzers,
)
from src.analysis.isc import compute_loo_isc  # noqa: E402
from src.definitions.constants import ExperimentNames, ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
%matplotlib inline

# Wavelet Phase Exploration

Exploratory analysis of **wavelet-transformed EEG data** (phase representation).

This notebook loads pre-computed (or freshly computed) wavelet phase transforms
in 4-D format `(n_subjects, n_channels, n_frequencies, n_times)` and runs the
following sketch analyses:

1. Phase distribution sanity check (circular histogram)
2. Inter-Trial Phase Coherence (ITPC) spectrum
3. Time–frequency ITPC map
4. Per-band ITPC time course
5. Phase-based ISC (cosine-similarity LOO-ISC on phase data) (cosine-similarity LOO-ISC on phase data)
6. ITPC vs. phase-ISC comparison

Phase values are in radians (−π, π].  Unlike power, phase is a **circular**
quantity, so standard linear statistics (mean, variance, Pearson *r*) are
replaced by circular analogues (circular mean, mean resultant length, ITPC).

See `README.md` in this directory for rationale and future extensions.

## Configuration

In [ ]:
# ── Experiment configuration ─────────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL, MusicTypeVariants.PSYTRANCE]
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False  # set True to re-process raw files

# ── Wavelet settings ─────────────────────────────────────────────────────────
REPRESENTATION = "phase"  # ← phase instead of power
WAVELET_FREQ_MIN = min(lo for lo, _ in FREQUENCY_BANDS.values())
WAVELET_FREQ_MAX = max(hi for _, hi in FREQUENCY_BANDS.values())
WAVELET_N_FREQS = max(
    2,
    int(
        round(
            (WAVELET_FREQ_MAX - WAVELET_FREQ_MIN)
            / _WAVELET_BAND_FREQ_RESOLUTION_HZ
        )
    )
    + 1,
)
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True  # -> (n_subjects, n_channels, n_freqs, n_times)

# ── Reuse / compute ──────────────────────────────────────────────────────────
REUSE_WAVELETS = True  # load from cache; set False to compute + save

# ── Compute-only subject subset ──────────────────────────────────────────────
N_SUBJECTS_SUBSET: int | None = 3

# ── Scope ─────────────────────────────────────────────────────────────────────
RUN_BROADBAND = True
RUN_PER_BAND = True

# ── Storage directory (same default as the CLI script) ────────────────────────
WAVELET_DIR: Path = (
    ProjectPaths.PROCESSED_DATA_DIR
    / ExperimentNames.PSILO_MUSIC.value
    / "wavelets"
)

# ── Plot saving ──────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR / "03-wavelet-analysis" / "plots" / "phase"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Wavelet base directory : {WAVELET_DIR}")
print(f"Frequencies            : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)")
print(f"Representation         : {REPRESENTATION}")
print(
    "Compute subset         : "
    + (
        f"{N_SUBJECTS_SUBSET} individuals"
        if not REUSE_WAVELETS and N_SUBJECTS_SUBSET is not None
        else "all individuals"
    )
)

## Data Loading

In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
)
datasets = analyzers_to_datasets(analyzers)

# Optional subject subset when computing from scratch
if not REUSE_WAVELETS and N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_SUBJECTS_SUBSET} individuals for compute (subset mode).")

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(f"  {label}: {ad.n_items} subjects, {ad.n_features} channels, {ad.n_samples} samples")

## Load or compute wavelet transforms

### Broadband

Stored in `WAVELET_DIR/broadband/`.

In [ ]:
broadband_datasets: dict = {}

if RUN_BROADBAND:
    broadband_datasets = _wavelet_transform(
        datasets=datasets,
        freqs=FREQS,
        representation=REPRESENTATION,
        keep_frequency_dim=KEEP_FREQUENCY_DIM,
        reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
        wavelet_dir=WAVELET_DIR / "broadband",
        reuse_wavelets=REUSE_WAVELETS,
    )
    for label, ad in broadband_datasets.items():
        source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
        print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

### Per-band

Each EEG band is stored in `WAVELET_DIR/band_<name>/`.

In [ ]:
band_datasets: dict[str, dict] = {}  # band_name -> {label: AnalysisData}

if RUN_PER_BAND:
    for band, (l_freq, h_freq) in FREQUENCY_BANDS.items():
        n_freqs = max(
            2,
            int(round((h_freq - l_freq) / _WAVELET_BAND_FREQ_RESOLUTION_HZ)) + 1,
        )
        band_freqs = np.linspace(l_freq, h_freq, n_freqs)
        band_datasets[band] = _wavelet_transform(
            datasets=datasets,
            freqs=band_freqs,
            representation=REPRESENTATION,
            keep_frequency_dim=KEEP_FREQUENCY_DIM,
            reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
            wavelet_dir=WAVELET_DIR / f"band_{band}",
            reuse_wavelets=REUSE_WAVELETS,
        )
        for label, ad in band_datasets[band].items():
            source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
            print(f"[{band:6s}] {label}: shape={ad.data.shape}  source={source}")

## Dataset Selection

Change `LABEL` to switch between music types.  The remaining cells use `bb_data`
(broadband phase, 4-D) and the derived scalars.

In [ ]:
LABEL = list(broadband_datasets.keys())[0]
# LABEL = list(broadband_datasets.keys())[1]  # uncomment for second music type

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

print(f"Dataset    : {LABEL}")
print(f"Shape      : {bb_data.shape}  (subjects × channels × freqs × times)")
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({n_freqs} steps)")
print(f"Phase range: [{bb_data.min():.2f}, {bb_data.max():.2f}] rad")

---
## 1 — Phase Distribution Sanity Check

Wavelet phase values should be approximately uniformly distributed over
(−π, π] when averaged across all channels, subjects, and frequencies.

A strongly non-uniform distribution would indicate artefacts or
data-processing issues.

In [ ]:
# Flatten all phase values for the histogram
all_phases = bb_data.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: linear histogram
axes[0].hist(all_phases, bins=64, density=True, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axhline(1.0 / (2 * np.pi), color="red", linestyle="--", label="Uniform density")
axes[0].set_xlabel("Phase (rad)")
axes[0].set_ylabel("Density")
axes[0].set_title(f"Phase distribution — {LABEL}")
axes[0].legend()

# Panel B: polar histogram
ax_polar = fig.add_subplot(122, projection="polar")
counts, bin_edges = np.histogram(all_phases, bins=64, range=(-np.pi, np.pi))
centres = 0.5 * (bin_edges[:-1] + bin_edges[1:])
widths = np.diff(bin_edges)
ax_polar.bar(centres, counts, width=widths, color="steelblue", alpha=0.7, edgecolor="white")
ax_polar.set_title(f"Polar phase histogram — {LABEL}", pad=15)

# Remove the Cartesian axes[1] since we replaced it with polar
axes[1].set_visible(False)

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "phase_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 2 — Inter-Trial Phase Coherence (ITPC) Spectrum

ITPC (also called Phase-Locking Value) measures consistency of phase angles
across subjects at each `(channel, frequency, time)` cell:

    ITPC(f, t) = |mean_over_subjects(exp(i * phase(f, t)))|

Values range from 0 (no consistency) to 1 (perfect phase alignment).

Averaging ITPC over time gives a **frequency-domain ITPC profile** showing
which frequencies exhibit the strongest cross-subject phase locking.

In [ ]:
# ITPC: mean resultant length across subjects
# bb_data: (n_subjects, n_channels, n_freqs, n_times)
complex_phases = np.exp(1j * bb_data)  # unit-magnitude complex phasors
itpc_full = np.abs(complex_phases.mean(axis=0))  # (n_channels, n_freqs, n_times)

# Average over time → (n_channels, n_freqs), then over channels → (n_freqs,)
itpc_ch_freq = itpc_full.mean(axis=2)  # (n_channels, n_freqs)
itpc_spectrum = itpc_ch_freq.mean(axis=0)  # (n_freqs,)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: ITPC spectrum
axes[0].plot(FREQS, itpc_spectrum, color="darkgreen", linewidth=1.5)
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("Mean ITPC")
axes[0].set_title(f"ITPC spectrum (time-averaged) — {LABEL}")
band_colors = {
    "delta": "#d4e6f1",
    "theta": "#d5f5e3",
    "alpha": "#fdebd0",
    "beta": "#fadbd8",
    "gamma": "#e8daef",
}
for band, (lo, hi) in FREQUENCY_BANDS.items():
    axes[0].axvspan(lo, hi, alpha=0.25, color=band_colors.get(band, "grey"), label=band)
axes[0].legend(fontsize=8, loc="upper right")

# Panel B: channel × frequency ITPC
im = axes[1].imshow(
    itpc_ch_freq,
    aspect="auto",
    origin="lower",
    extent=[FREQS[0], FREQS[-1], 0, n_channels],
    cmap="YlGn",
    vmin=0,
)
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("Channel index")
axes[1].set_title(f"Channel × frequency ITPC — {LABEL}")
fig.colorbar(im, ax=axes[1], label="ITPC")

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "itpc_spectrum.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 3 — Time–Frequency ITPC Map

Channel-averaged ITPC as a 2-D image `(n_freqs × n_times)`.

Reveals **when** and at **which frequencies** subjects' phases align most
strongly — e.g. stimulus-locked phase resets at musical transitions.

In [ ]:
# Channel-averaged ITPC → (n_freqs, n_times)
itpc_tf = itpc_full.mean(axis=0)  # (n_freqs, n_times)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(
    itpc_tf,
    aspect="auto",
    origin="lower",
    extent=[time[0], time[-1], FREQS[0], FREQS[-1]],
    cmap="YlGn",
    vmin=0,
)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Frequency (Hz)")
ax.set_title(f"Time–frequency ITPC map — {LABEL}")
fig.colorbar(im, ax=ax, label="ITPC")

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "time_frequency_itpc.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4 — Per-Band ITPC Time Course

For each frequency band, average ITPC over the band's frequency bins, then
average over channels to get a single ITPC time course.

Directly comparable to the per-band power time course in the power notebook
but now showing **phase consistency** instead of amplitude.

In [ ]:
fig, axes = plt.subplots(
    len(FREQUENCY_BANDS), 1,
    figsize=(14, 3 * len(FREQUENCY_BANDS)),
    sharex=True,
)
if len(FREQUENCY_BANDS) == 1:
    axes = [axes]

for ax, (band, (lo, hi)) in zip(axes, FREQUENCY_BANDS.items()):
    band_mask = (FREQS >= lo) & (FREQS <= hi)
    # Band-averaged ITPC → (n_channels, n_times)
    band_itpc = itpc_full[:, band_mask, :].mean(axis=1)
    # Channel average → (n_times,)
    band_itpc_avg = band_itpc.mean(axis=0)

    ax.plot(time, band_itpc_avg, color="darkgreen", linewidth=0.8)
    ax.set_ylabel("ITPC")
    ax.set_title(f"{band} ({lo:.0f}–{hi:.0f} Hz)")
    ax.set_ylim(bottom=0)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"Per-band ITPC time course — {LABEL}", fontsize=13, y=1.01)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "band_itpc_timecourse.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 5 — Phase-Based ISC (Per Band)

To complement the ITPC analysis (which measures consistency of the raw phase
angle), we also compute leave-one-out ISC on the **cosine of the phase**.

`cos(phase)` maps the circular phase to a linear signal (−1, 1) that captures
the real component of the oscillation.  Applying standard Pearson LOO-ISC to
`cos(phase)` tests whether subjects share the same oscillatory *timing*,
independently of amplitude.

This mirrors the wavelet-power LOO-ISC in `wavelet_power_exploration.ipynb`
but operates on phase-derived features.

In [ ]:
band_mean_iscs: dict[str, np.ndarray] = {}

for band, (lo, hi) in FREQUENCY_BANDS.items():
    band_mask = (FREQS >= lo) & (FREQS <= hi)
    # Band-average phase → (n_subjects, n_channels, n_times)
    band_phase_3d = circmean(
        bb_data[:, :, band_mask, :], high=np.pi, low=-np.pi, axis=2
    )
    # Convert circular phase to linear feature via cosine
    cos_phase_3d = np.cos(band_phase_3d)
    _, mean_loo_isc = compute_loo_isc(cos_phase_3d)
    band_mean_iscs[band] = mean_loo_isc  # (n_channels,)
    print(
        f"  {band:6s}  mean LOO-ISC(cos φ) = {mean_loo_isc.mean():.4f}"
        f"  (median = {np.median(mean_loo_isc):.4f})"
    )

In [ ]:
bands_list = list(band_mean_iscs.keys())
means = [band_mean_iscs[b].mean() for b in bands_list]
medians = [np.median(band_mean_iscs[b]) for b in bands_list]

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(bands_list))
width = 0.35
ax.bar(x - width / 2, means, width, label="Mean", color="darkgreen")
ax.bar(x + width / 2, medians, width, label="Median", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(bands_list)
ax.set_ylabel("LOO-ISC on cos(phase)")
ax.set_title(f"Per-band phase LOO-ISC — {LABEL}")
ax.legend()

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "phase_loo_isc_bar.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.1 — Phase LOO-ISC distribution per band

Histogram of per-channel LOO-ISC(cos φ) for each band, clipped at the
99th percentile.

In [ ]:
fig, axes = plt.subplots(
    1, len(FREQUENCY_BANDS),
    figsize=(3.5 * len(FREQUENCY_BANDS), 4),
    sharey=True,
)
if len(FREQUENCY_BANDS) == 1:
    axes = [axes]

for ax, band in zip(axes, FREQUENCY_BANDS):
    vals = band_mean_iscs[band]
    clip = np.percentile(vals, 99)
    ax.hist(
        vals[vals <= clip], bins=30,
        color="darkgreen", edgecolor="white", alpha=0.8,
    )
    ax.axvline(
        vals.mean(), color="red", linestyle="--", linewidth=1,
        label=f"mean={vals.mean():.3f}",
    )
    ax.set_xlabel("LOO-ISC(cos φ)")
    ax.set_title(band)
    ax.legend(fontsize=7)

axes[0].set_ylabel("Number of channels")
fig.suptitle(f"Phase LOO-ISC distributions — {LABEL}", fontsize=13, y=1.02)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "phase_loo_isc_distributions.png",
        dpi=150, bbox_inches="tight",
    )
plt.show()

---
## 6 — ITPC vs. Phase-ISC Comparison

Side-by-side comparison of the two phase-synchrony measures per band:
- **ITPC** (mean resultant length) — measures *raw* phase concentration
- **LOO-ISC(cos φ)** — measures *pairwise* temporal alignment via cosine

Bands where both metrics are elevated indicate robust phase synchrony.

In [ ]:
# Compute per-band mean ITPC (time- and channel-averaged)
band_mean_itpc: dict[str, float] = {}
for band, (lo, hi) in FREQUENCY_BANDS.items():
    band_mask = (FREQS >= lo) & (FREQS <= hi)
    band_itpc_val = itpc_full[:, band_mask, :].mean()
    band_mean_itpc[band] = float(band_itpc_val)

bands_list = list(FREQUENCY_BANDS.keys())
itpc_vals = [band_mean_itpc[b] for b in bands_list]
isc_vals = [band_mean_iscs[b].mean() for b in bands_list]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: grouped bar
x = np.arange(len(bands_list))
width = 0.35
axes[0].bar(x - width / 2, itpc_vals, width, label="ITPC", color="darkgreen")
axes[0].bar(x + width / 2, isc_vals, width, label="ISC(cos φ)", color="steelblue")
axes[0].set_xticks(x)
axes[0].set_xticklabels(bands_list)
axes[0].set_ylabel("Score")
axes[0].set_title(f"ITPC vs Phase-ISC — {LABEL}")
axes[0].legend()

# Panel B: scatter
axes[1].scatter(itpc_vals, isc_vals, s=80, color="purple", zorder=3)
for i, band in enumerate(bands_list):
    axes[1].annotate(band, (itpc_vals[i], isc_vals[i]), fontsize=9, ha="left", va="bottom")
axes[1].set_xlabel("Mean ITPC")
axes[1].set_ylabel("Mean LOO-ISC(cos φ)")
axes[1].set_title(f"ITPC–ISC relationship — {LABEL}")

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "itpc_vs_isc.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Summary

The arrays are available for further analysis:

- `broadband_datasets[LABEL].data` — 4-D broadband wavelet phase
- `band_datasets[band][LABEL].data` — 4-D per-band wavelet phase
- `itpc_full` — `(n_channels, n_freqs, n_times)` ITPC map
- `band_mean_iscs[band]` — per-channel LOO-ISC(cos φ) for each band
- `band_mean_itpc[band]` — scalar mean ITPC for each band

See `README.md` in this directory for ideas on extending these analyses
(condition comparison, topographic mapping, time–frequency ISC, etc.).